# Başlanğıc RAG (Retrieval-Augmented Generation) — Hugging Face ilə

Bu notebook sıfırdan sadə bir RAG sistemi qurur. Hər şey — embedding modeli də, cavab yazan model də — Hugging Face-dən lokal yüklənir. Xarici API açarı lazım deyil, pulsuzdur.

## Retrieval (axtarış) nədir?

RAG-ın "R" hərfi buradan gəlir (Retrieval-Augmented Generation). Model sualı cavablandırmazdan əvvəl, əlaqəli sənədləri böyük bir kolleksiyadan tapır ("retrieve" edir) və bunları modelə əlavə kontekst kimi verir.

Bu niyə lazımdır: dil modelləri öz təlim datasında olmayan və ya köhnəlmiş məlumatı bilmir. Retrieval bu boşluğu doldurur — modelin cavabını sənin öz sənədlərinə (PDF, daxili sənədlər, wiki və s.) əsaslandırmasına imkan verir və uydurma (hallucination) ehtimalını azaldır.


## Embedding nədir?

Embedding — mətni (söz, cümlə, paraqraf) ədədi vektora çevirmə prosesidir. Mənaca yaxın mətnlərin vektorları da bir-birinə yaxın olur. Məsələn "pişik" və "pişik balası" vektorları yaxın olacaq, "pişik" və "avtomobil" isə uzaq.

RAG-da embedding belə istifadə olunur:
1. Bütün sənədlər əvvəlcədən vektora çevrilir və saxlanılır (vektor bazası / indeks).
2. İstifadəçi sual verəndə, sualın özü də eyni modeldə vektora çevrilir.
3. Sual vektoruna ən yaxın sənəd vektorları tapılır (məsafə ölçüsü ilə, məs. cosine ya L2) — bunlar ən əlaqəli sənədlərdir.

Hər iki notebookda embedding üçün `sentence-transformers` kitabxanasından `all-MiniLM-L6-v2` modeli istifadə olunur — kiçik (~80MB), sürətli və CPU-da belə rahat işləyir.


## Bu notebookda addımlar

1. Kitabxanaların qurulması
2. Nümunə sənədlər (kiçik korpus)
3. Sənədləri embedding-ə çevirmək
4. FAISS ilə vektor indeksi qurmaq
5. Retrieval funksiyası
6. Hugging Face-dən kiçik generasiya modeli yükləmək
7. RAG pipeline: retrieval + generasiya
8. Test sualları

> Qeyd: GPU olmadan da işləyir, amma Colab-da Runtime → Change runtime type → GPU seçsən generasiya addımı sürətlənər.

In [ ]:
!pip install -q sentence-transformers faiss-cpu transformers accelerate

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import pipeline

### 1. Nümunə sənədlər
Realda bunlar PDF, daxili sənədlər, dataset sətirləri və s. ola bilər. Sadəlik üçün bir neçə qısa mətn paraqrafı istifadə edirik.

In [ ]:
documents = [
    "The Eiffel Tower is a wrought-iron lattice tower located in Paris, France. It was completed in 1889 and stands about 330 meters tall.",
    "Photosynthesis is the process by which green plants use sunlight to synthesize food from carbon dioxide and water.",
    "Python is a high-level, interpreted programming language known for its readability and wide use in data science and AI.",
    "The Great Wall of China is a series of fortifications built to protect Chinese states from invasions, stretching thousands of kilometers.",
    "Machine learning is a subset of artificial intelligence where systems learn patterns from data rather than being explicitly programmed.",
    "The human heart pumps blood through the circulatory system, delivering oxygen and nutrients to tissues throughout the body.",
]
print(f"{len(documents)} sənəd yükləndi")


### 2. Embedding modelini yükləmək və sənədləri kodlaşdırmaq

In [ ]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = embedding_model.encode(documents, convert_to_numpy=True)
print("Embedding ölçüsü:", doc_embeddings.shape)  # (sənəd sayı, vektor ölçüsü)


### 3. FAISS vektor indeksi
FAISS (Facebook AI Similarity Search) — vektorlar arasında sürətli oxşarlıq axtarışı üçün kitabxanadır. `IndexFlatL2` ən sadə variantdır — bütün vektorları düz saxlayır və məsafəni birbaşa hesablayır (kiçik data üçün kifayətdir).

In [ ]:
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)
print(f"İndeksdə {index.ntotal} sənəd var, hər biri {dimension} ölçülü vektor")


### 4. Retrieval funksiyası
Verilən sual üçün ən yaxın `k` sənədi tapır.

In [ ]:
def retrieve(query, k=2):
    query_embedding = embedding_model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, k)
    return [documents[i] for i in indices[0]]

# sürətli yoxlama
retrieve("Which language is popular in AI?")


### 5. Generasiya modeli

Retrieval-la tapılan kontekst + sual birlikdə kiçik instruct modelə verilir ki, cavab yaza bilsin. `flan-t5-base` seçilib, çünki kiçikdir (~250M parametr) və CPU-da belə rahat işləyir. Daha güclü cavab üçün `Qwen/Qwen2.5-1.5B-Instruct` kimi modelə keçmək olar (GPU tövsiyə olunur).

In [ ]:
generator = pipeline("text2text-generation", model="google/flan-t5-base")

### 6. RAG pipeline — retrieval + generasiya birləşməsi

In [ ]:
def rag_answer(query, k=2):
    context_docs = retrieve(query, k=k)
    context = "\n".join(context_docs)
    prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
    result = generator(prompt, max_length=100)
    return result[0]["generated_text"], context_docs

answer, sources = rag_answer("What is Python used for?")
print("Cavab:", answer)
print("\nİstifadə olunan mənbələr:")
for s in sources:
    print("-", s)

### 7. Bir neçə test sualı

In [ ]:
for q in [
    "How tall is the Eiffel Tower?",
    "What does the heart do?",
    "What is machine learning?",
]:
    ans, _ = rag_answer(q)
    print(f"Sual: {q}\nCavab: {ans}\n")

## Növbəti addımlar

- Öz sənədlərini (PDF, .txt, wiki export) yükləyib `documents` siyahısını onlarla əvəz et
- Uzun sənədləri kiçik hissələrə bölmək (chunking) strategiyalarını araşdır — bir paraqraf çox uzun olanda embedding keyfiyyəti aşağı düşür
- `IndexFlatL2` əvəzinə böyük data üçün `IndexIVFFlat` kimi sürətli FAISS indekslərinə bax
- FAISS əvəzinə hazır vektor bazalarını (Chroma, Qdrant, Pinecone) sınamaq
- Retrieval keyfiyyətini ölçmək üçün "hansı sənəd hansı suala aid olmalıdır" siyahısı hazırlayıb dəqiqliyi hesablamaq
